# 🧠 Advanced Vietnamese RAG System (AI VIET NAM Course Project)

Notebook này chứa toàn bộ quy trình triển khai hệ thống **Hỏi Đáp Tài Liệu Tiếng Việt Nâng Cao (Advanced RAG)** theo đúng tài liệu hướng dẫn thực nghiệm của khóa học AI VIET NAM.

### 🔄 Quy trình Kiểm soát & Tiếp cận Token 4 Giai đoạn:
1. **Ingestion (Nạp dữ liệu)**: Semantic Chunker tách câu tiếng Việt (Underthesea), giới hạn tokens cứng để bảo vệ LLM.
2. **Routing (Định tuyến)**: Router siêu nhẹ phân loại intent câu hỏi sang Local Search, Web Search, hoặc kết hợp (Hybrid).
3. **Retrieval & Context Compression (Truy xuất & Nén ngữ cảnh)**: Hybrid Search (Chroma DB + BM25 với RRF/Interleaving) -> Reranking (Cross-Encoder/MMR) cắt tỉa chỉ giữ lại Top-K chunks tốt nhất.
4. **Generation & Post-processing (Sinh & Hậu xử lý)**: Prompt cấu trúc chặt chẽ -> LLM trả lời ngắn gọn -> Hậu xử lý loại bỏ từ thừa dông dài.

---



### 🛠️ Bước 1: Cài đặt các thư viện cần thiết


In [ ]:
!pip install -q \
    "langchain" \
    "langchain-community" \
    "langchain-chroma" \
    "langchain-google-genai" \
    "langchain-openai" \
    "langchain-huggingface" \
    "underthesea" \
    "rank-bm25" \
    "duckduckgo-search" \
    "gradio" \
    "ragas" \
    "datasets" \
    "sentence-transformers" \
    "transformers>=4.51.0" \
    "torch" \
    "accelerate" \
    "python-docx" \
    "pypdf" \
    "pandas" \
    "matplotlib" \
    "seaborn" \
    "nest-asyncio" 


### ⚙️ Bước 2: Khai báo thư viện và thiết lập cấu hình cơ bản


In [ ]:
import os
import re
import sys
import time
import json
import random
import hashlib
import shutil
import unicodedata
import numpy as np
import pandas as pd
import nest_asyncio
import asyncio
from typing import List, Dict, Any, Tuple, Optional
from tqdm import tqdm
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.prompts import PromptTemplate

# Kích hoạt nest_asyncio để cho phép chạy event loop lặp trong môi trường Jupyter/Colab
nest_asyncio.apply()

# Cấu hình bản vá động tương thích ngược cho asyncio.run với loop_factory (khắc phục xung đột uvicorn trên Python 3.11+)
try:
    _orig_run = asyncio.run
    def _safe_run(main, *, debug=False, loop_factory=None):
        try:
            if loop_factory is not None:
                # Sử dụng loop_factory để tạo loop mới
                loop = loop_factory()
                asyncio.set_event_loop(loop)
            # Chuyển tiếp an toàn, loại bỏ loop_factory để nest_asyncio không gặp lỗi signature
            return _orig_run(main, debug=debug)
        except RuntimeError as e:
            if "already running" in str(e):
                loop = asyncio.get_event_loop()
                return loop.run_until_complete(main)
            raise e
    asyncio.run = _safe_run
except Exception as e:
    print(f"Cảnh báo: Không thể vá lỗi asyncio.run: {e}")

# Cấu hình seed ngẫu nhiên để thực nghiệm được nhất quán
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Cấu hình thư mục lưu trữ dữ liệu
ROOT_DIR = "./"
DOC_DIR = os.path.join(ROOT_DIR, "Ducument")
CHROMA_DIR = os.path.join(ROOT_DIR, "chroma_db")
LOG_DIR = os.path.join(ROOT_DIR, "logs")
MODELS_DIR = os.path.join(ROOT_DIR, "models")

for d in [DOC_DIR, CHROMA_DIR, LOG_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
    
print("Đã thiết lập môi trường và cấu hình các thư mục lưu trữ.")


### 📥 Bước 3: Triển khai SimpleLoader (Nạp & Làm sạch dữ liệu tiếng Việt)
Hỗ trợ file PDF và DOCX. Chuẩn hóa font tiếng Việt NFC và gán nhãn tag để phục vụ Pre-filtering.


In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
import docx

def clean_vietnamese_text(text: str) -> str:
    """
    Làm sạch văn bản tiếng Việt:
    1. Chuẩn hóa Unicode NFC để tránh lỗi font chữ tiếng Việt.
    2. Loại bỏ các ký tự điều khiển lạ.
    3. Gộp các khoảng trắng và các dòng trống liên tiếp.
    """
    if not text:
        return ""
    text = unicodedata.normalize('NFC', text)
    text = "".join(
        char for char in text
        if not unicodedata.category(char).startswith('C') or char in '\n\t'
    )
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()

class SimpleLoader:
    def load_pdf(self, file_path: str) -> List[Document]:
        if not os.path.exists(file_path):
            return []
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()
            for doc in docs:
                doc.page_content = clean_vietnamese_text(doc.page_content)
                doc.metadata["source_name"] = os.path.basename(file_path)
            return docs
        except Exception as e:
            print(f"[Lỗi] Nạp PDF {file_path}: {str(e)}")
            return []
            
    def load_docx(self, file_path: str) -> List[Document]:
        if not os.path.exists(file_path):
            return []
        try:
            doc_obj = docx.Document(file_path)
            full_text = []
            for para in doc_obj.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            for table in doc_obj.tables:
                for row in table.rows:
                    row_text = [cell.text.strip() for cell in row.cells if cell.text.strip()]
                    if row_text:
                        full_text.append(" | ".join(row_text))
            text_content = "\n".join(full_text)
            cleaned_content = clean_vietnamese_text(text_content)
            doc = Document(
                page_content=cleaned_content,
                metadata={
                    "source": file_path,
                    "source_name": os.path.basename(file_path),
                    "page": 1
                }
            )
            return [doc]
        except Exception as e:
            print(f"[Lỗi] Nạp DOCX {file_path}: {str(e)}")
            return []
            
    def load_file(self, file_path: str, tag: str = None) -> List[Document]:
        ext = os.path.splitext(file_path)[1].lower()
        docs = []
        if ext == '.pdf':
            docs = self.load_pdf(file_path)
        elif ext == '.docx':
            docs = self.load_docx(file_path)
        
        if tag:
            for doc in docs:
                doc.metadata["doc_tag"] = tag
        return docs
        
    def load_directory(self, dir_path: str, tag: str = None) -> List[Document]:
        if not os.path.isdir(dir_path):
            return []
        all_docs = []
        import glob
        pdf_files = glob.glob(os.path.join(dir_path, "*.pdf"))
        docx_files = glob.glob(os.path.join(dir_path, "*.docx"))
        all_files = pdf_files + docx_files
        
        if not tag:
            tag = os.path.basename(os.path.normpath(dir_path))
            
        print(f"Bắt đầu nạp thư mục '{dir_path}' với nhãn tag='{tag}' ({len(all_files)} files)...")
        for file_path in all_files:
            docs = self.load_file(file_path, tag=tag)
            all_docs.extend(docs)
        return all_docs


### 📏 Bước 4: Triển khai Semantic Chunker (Stage 1 Token Optimization)
Tính cosine similarity của các câu kề nhau bằng embedding model. Enforce giới hạn token bằng `tiktoken` để tránh OOM/timeout.


In [ ]:
import tiktoken
from underthesea import sent_tokenize

try:
    _tokenizer = tiktoken.get_encoding("cl100k_base")
except Exception:
    _tokenizer = None

def count_tokens(text: str) -> int:
    if not text:
        return 0
    if _tokenizer:
        return len(_tokenizer.encode(text, disallowed_special=()))
    return len(text.split())

class FixedSizeChunker:
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 200):
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        self.splitter = RecursiveCharacterTextSplitter(
            separators=["\n\n", "\n", " ", ""],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len
        )
    def split(self, documents: List[Document]) -> List[Document]:
        return self.splitter.split_documents(documents)

class SemanticChunker:
    def __init__(
        self, 
        embedding_model: Any, 
        breakpoint_threshold: float = 0.5,
        min_chunk_tokens: int = 100,
        max_chunk_tokens: int = 350,
        overlap_chars: int = 128
    ):
        self.embeddings = embedding_model
        self.breakpoint_threshold = breakpoint_threshold
        self.min_chunk_tokens = min_chunk_tokens
        self.max_chunk_tokens = max_chunk_tokens
        self.overlap_chars = overlap_chars

    def _calculate_cosine_similarity(self, emb1: List[float], emb2: List[float]) -> float:
        v1, v2 = np.array(emb1), np.array(emb2)
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return float(np.dot(v1, v2) / (norm1 * norm2))

    def _split_into_sentences(self, text: str) -> List[str]:
        sentences = sent_tokenize(text)
        return [s.strip() for s in sentences if s.strip() and len(s) > 15]

    def _chunk_by_semantic_similarity(self, sentences: List[str]) -> List[str]:
        if not sentences:
            return []

        # Gọi API embedding một lần duy nhất cho toàn bộ các câu trong văn bản để đảm bảo tính liên tục ngữ nghĩa
        try:
            sentence_embeddings = self.embeddings.embed_documents(sentences)
        except Exception as e:
            print(f"[Cảnh báo] Lỗi gọi embedding cho chunking: {str(e)}. Fallback sang gộp cố định.")
            chunks = []
            for i in range(0, len(sentences), 5):
                chunks.append(" ".join(sentences[i:i+5]))
            return chunks

        chunks = []
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            prev_emb = sentence_embeddings[i-1]
            curr_emb = sentence_embeddings[i]
            similarity = self._calculate_cosine_similarity(prev_emb, curr_emb)
            
            chunk_text = " ".join(current_chunk)
            chunk_tokens = count_tokens(chunk_text)
            next_sentence_tokens = count_tokens(sentences[i])
            
            # ĐIỀU KIỆN 1: Nếu vượt quá giới hạn trần (max_chunk_tokens), bắt buộc ngắt chunk
            if chunk_tokens + next_sentence_tokens >= self.max_chunk_tokens:
                chunks.append(chunk_text)
                current_chunk = [sentences[i]]
            # ĐIỀU KIỆN 2: Nếu chưa đạt giới hạn sàn (min_chunk_tokens), gộp tiếp bất kể tương đồng
            elif chunk_tokens < self.min_chunk_tokens:
                current_chunk.append(sentences[i])
            # ĐIỀU KIỆN 3: Đã đủ kích thước tối thiểu, kiểm tra tương đồng ngữ nghĩa
            elif similarity >= self.breakpoint_threshold:
                current_chunk.append(sentences[i])
            # ĐIỀU KIỆN 4: Chuyển chủ đề ngữ nghĩa -> ngắt chunk, tạo nhóm mới
            else:
                chunks.append(chunk_text)
                current_chunk = [sentences[i]]
                
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        return chunks

    def split(self, documents: List[Document]) -> List[Document]:
        all_chunks = []
        for doc in documents:
            content = doc.page_content
            if not content.strip():
                continue
            sentences = self._split_into_sentences(content)
            if not sentences:
                continue
            chunks_text = self._chunk_by_semantic_similarity(sentences)
            total_chunks = len(chunks_text)
            
            for idx, chunk_text in enumerate(chunks_text):
                if not chunk_text.strip():
                    continue
                # Overlap Handling (Gối đầu ngữ cảnh tính theo ký tự)
                if idx > 0 and self.overlap_chars > 0:
                    prev_chunk = chunks_text[idx-1]
                    if len(prev_chunk) > self.overlap_chars:
                        overlap = prev_chunk[-self.overlap_chars:]
                        chunk_text = overlap + " " + chunk_text
                
                chunk_metadata = doc.metadata.copy()
                chunk_metadata["chunk_index"] = idx
                chunk_metadata["total_chunks"] = total_chunks
                
                chunk_doc = Document(
                    page_content=chunk_text,
                    metadata=chunk_metadata
                )
                all_chunks.append(chunk_doc)
        return all_chunks


### 🔍 Bước 5: Triển khai Vector DB, BM25 và Hybrid Search (RRF & Interleaving)
Tích hợp `word_tokenize` Việt hóa cho BM25. Trình bày thuật toán Reciprocal Rank Fusion (RRF) dùng mã băm MD5 để tối ưu hóa RAM.


In [ ]:
from langchain_chroma import Chroma
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from rank_bm25 import BM25Okapi
from underthesea import word_tokenize
from duckduckgo_search import DDGS

class VectorStoreManager:
    def __init__(self, persist_directory: str = "./chroma_db", embedding_model: Any = None):
        self.persist_directory = persist_directory
        self.embeddings = embedding_model
        self.db = None
        
    def init_db(self, documents: List[Document] = None):
        if documents and len(documents) > 0:
            print(f"Khởi tạo VectorDB mới tại '{self.persist_directory}' với {len(documents)} chunks...")
            self.db = Chroma.from_documents(
                documents=documents,
                embedding=self.embeddings,
                persist_directory=self.persist_directory,
                collection_name="advanced_rag_collection"
            )
        else:
            print(f"Nạp VectorDB đã có từ thư mục '{self.persist_directory}'...")
            self.db = Chroma(
                persist_directory=self.persist_directory,
                embedding_function=self.embeddings,
                collection_name="advanced_rag_collection"
            )
            
    def get_retriever(self, k: int = 5, doc_tags: List[str] = None) -> BaseRetriever:
        if not self.db:
            raise ValueError("VectorDB chưa được khởi tạo.")
        search_kwargs = {"k": k}
        if doc_tags and len(doc_tags) > 0:
            if len(doc_tags) == 1:
                search_kwargs["filter"] = {"doc_tag": doc_tags[0]}
            else:
                search_kwargs["filter"] = {"doc_tag": {"$in": doc_tags}}
        return self.db.as_retriever(
            search_type="similarity",
            search_kwargs=search_kwargs
        )

class BM25Retriever(BaseRetriever):
    documents: List[Document]
    bm25: Any
    k: int = 5

    class Config:
        arbitrary_types_allowed = True

    @classmethod
    def from_documents(cls, documents: List[Document], k: int = 5):
        tokenized_docs = [
            word_tokenize(doc.page_content.lower())
            for doc in documents
        ]
        bm25 = BM25Okapi(tokenized_docs)
        return cls(documents=documents, bm25=bm25, k=k)

    def _get_relevant_documents(
        self, query: str, *, run_manager: Optional[CallbackManagerForRetrieverRun] = None
    ) -> List[Document]:
        tokenized_query = word_tokenize(query.lower())
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = np.argsort(scores)[::-1][:self.k]
        
        results = []
        for idx in top_k_indices:
            if scores[idx] > 0:
                results.append(self.documents[idx])
        return results

class HybridRetriever(BaseRetriever):
    bm25_retriever: BM25Retriever
    vector_retriever: BaseRetriever
    top_k: int = 5
    fusion_mode: str = "rrf"
    rrf_k: int = 60

    class Config:
        arbitrary_types_allowed = True

    def _get_doc_id(self, doc: Document) -> str:
        return hashlib.md5(doc.page_content.encode("utf-8")).hexdigest()

    def _reciprocal_rank_fusion(self, bm25_docs: List[Document], vector_docs: List[Document]) -> List[Document]:
        from collections import defaultdict
        rrf_scores = defaultdict(float)
        doc_map = {}
        
        for rank, doc in enumerate(bm25_docs, start=1):
            doc_id = self._get_doc_id(doc)
            rrf_scores[doc_id] += 1.0 / (self.rrf_k + rank)
            doc_map[doc_id] = doc
            
        for rank, doc in enumerate(vector_docs, start=1):
            doc_id = self._get_doc_id(doc)
            rrf_scores[doc_id] += 1.0 / (self.rrf_k + rank)
            doc_map[doc_id] = doc
            
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [doc_map[doc_id] for doc_id, _ in sorted_docs]

    def _interleave_results(self, bm25_docs: List[Document], vector_docs: List[Document]) -> List[Document]:
        merged = []
        seen = set()
        max_len = max(len(bm25_docs), len(vector_docs))
        for i in range(max_len):
            if i < len(bm25_docs):
                doc = bm25_docs[i]
                if doc.page_content not in seen:
                    merged.append(doc)
                    seen.add(doc.page_content)
                if len(merged) >= self.top_k:
                    break
            if i < len(vector_docs):
                doc = vector_docs[i]
                if doc.page_content not in seen:
                    merged.append(doc)
                    seen.add(doc.page_content)
                if len(merged) >= self.top_k:
                    break
        return merged

    def _get_relevant_documents(
        self, query: str, *, run_manager: Optional[CallbackManagerForRetrieverRun] = None
    ) -> List[Document]:
        bm25_results = self.bm25_retriever.invoke(query)
        vector_results = self.vector_retriever.invoke(query)
        
        if self.fusion_mode == "rrf":
            fused = self._reciprocal_rank_fusion(bm25_results, vector_results)
        else:
            fused = self._interleave_results(bm25_results, vector_results)
        return fused[:self.top_k]

class WebSearchRetriever(BaseRetriever):
    max_results: int = 5
    timeout_seconds: int = 10

    def _get_relevant_documents(
        self, query: str, *, run_manager: Optional[CallbackManagerForRetrieverRun] = None
    ) -> List[Document]:
        print(f"[Internet Search] Đang tra cứu web: '{query}'...")
        web_docs = []
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=self.max_results))
            for idx, r in enumerate(results):
                title = r.get("title", "Tin tức Web")
                link = r.get("href", "http://duckduckgo.com")
                snippet = r.get("body", "")
                if snippet and snippet.strip():
                    display_title = title[:50] + "..." if len(title) > 50 else title
                    doc = Document(
                        page_content=snippet,
                        metadata={
                            "source": link,
                            "source_name": f"Web: {display_title}",
                            "title": title,
                            "page": 1,
                            "doc_tag": "Internet"
                        }
                    )
                    web_docs.append(doc)
        except Exception as e:
            print(f"[Internet Search] Lỗi gọi DuckDuckGo: {str(e)}")
        return web_docs


### 🔀 Bước 6: Triển khai Query Router & Query Transformers (HyDE, Decomposition)
**Stage 2 Routing & Query Transformation**. Router siêu nhẹ ép định dạng JSON để tiết kiệm Output Tokens. HyDE sinh tài liệu giả định, Decomposition phân rã tối đa 3 câu hỏi phụ.


In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.prompts import PromptTemplate

class QueryRouter:
    def __init__(self, llm: BaseChatModel):
        self.llm = llm
        self.prompt_template = """Phân loại câu hỏi của người dùng vào một trong 3 nhóm nguồn thông tin:
- "local": Tài liệu chuyên ngành về AI, lập trình, hoặc các bộ Luật Lao động, Luật Đất đai Việt Nam.
- "web": Câu hỏi về tin tức thời sự cập nhật, thời tiết hôm nay, sự kiện mới diễn ra hoặc thông tin đời sống chung.
- "hybrid": Câu hỏi cần đối chiếu dữ liệu nội bộ đồng thời kiểm tra thêm thông tin cập nhật ngoài internet.

CHỈ trả về kết quả dưới định dạng JSON duy nhất như sau, tuyệt đối không viết thêm bất kỳ lời giải thích nào:
{{"route": "local" | "web" | "hybrid"}}

Câu hỏi của User: "{query}"
JSON Output:"""
        self.prompt = PromptTemplate.from_template(self.prompt_template)
        
    def route_query(self, query: str) -> str:
        formatted_prompt = self.prompt.format(query=query)
        try:
            response = self.llm.invoke(formatted_prompt)
            response_text = response.content.strip()
            if response_text.startswith("```json"):
                response_text = response_text[7:]
            if response_text.endswith("```"):
                response_text = response_text[:-3]
            response_text = response_text.strip()
            
            data = json.loads(response_text)
            route = data.get("route", "local").lower().strip()
            if route in ["local", "web", "hybrid"]:
                return route
            return "local"
        except Exception as e:
            print(f"[Router] Lỗi parse JSON: {str(e)}. Dùng Regex fallback.")
            text = response.content.strip() if 'response' in locals() else ""
            if "local" in text.lower():
                return "local"
            elif "web" in text.lower():
                return "web"
            elif "hybrid" in text.lower():
                return "hybrid"
            return "local"

class HyDEQueryTransformer:
    def __init__(self, llm: BaseChatModel):
        self.llm = llm
        self.prompt_template = """Hãy viết một đoạn văn ngắn (khoảng 100 từ) trả lời giả định cho câu hỏi dưới đây.
Viết như thể đây là một đoạn trích dẫn chuyên môn từ sách giáo khoa hoặc tài liệu hướng dẫn.
Không cần bắt đầu bằng những câu như "Theo tôi...", "Câu trả lời giả định...", hãy đi thẳng vào nội dung chính.

Câu hỏi: {question}
Đoạn văn giả định:"""
        self.prompt = PromptTemplate.from_template(self.prompt_template)
        
    def generate_hypothetical_doc(self, question: str) -> str:
        formatted_prompt = self.prompt.format(question=question)
        try:
            response = self.llm.invoke(formatted_prompt)
            return response.content.strip()
        except Exception as e:
            print(f"[HyDE] Lỗi: {str(e)}")
            return question

class DecompositionQueryTransformer:
    def __init__(self, llm: BaseChatModel):
        self.llm = llm
        self.prompt_template = """Hãy phân tích và tách câu hỏi gốc dưới đây thành tối đa 3 câu hỏi phụ (sub-queries) đơn giản, mỗi câu tập trung vào một khía cạnh riêng biệt.
Nếu câu hỏi đã đơn giản, chỉ cần trả về chính câu hỏi đó.
Mỗi câu hỏi phụ viết trên một dòng mới. Không viết số thứ tự, không kèm ký tự đặc biệt ở đầu dòng.

Câu hỏi gốc: {question}
Các câu hỏi phụ:"""
        self.prompt = PromptTemplate.from_template(self.prompt_template)
        
    def decompose(self, question: str) -> List[str]:
        formatted_prompt = self.prompt.format(question=question)
        try:
            response = self.llm.invoke(formatted_prompt)
            lines = response.content.strip().split("\n")
            sub_queries = []
            for line in lines:
                cleaned_line = line.strip()
                cleaned_line = re.sub(r"^[\d\.\-\*\•\s]+", "", cleaned_line).strip()
                if cleaned_line and len(cleaned_line) > 5:
                    sub_queries.append(cleaned_line)
            return sub_queries[:3]
        except Exception as e:
            print(f"[Decomposition] Lỗi: {str(e)}")
            return [question]


### ⚡ Bước 7: Triển khai Rerankers (Stage 3 Context Compression)
Sử dụng mô hình Cross-Encoder cục bộ Qwen-Reranker (an toàn cho cả CPU/GPU) và Maximal Marginal Relevance (MMR). Cắt tỉa Top 15-20 candidates xuống Top 3 Chunks vàng.


In [ ]:
# Chỉ sử dụng cho Cross-Encoder
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
except ImportError:
    torch = None

class CrossEncoderReranker:
    def __init__(self, model_name: str = "Qwen/Qwen3-Reranker-0.6B", top_k: int = 3):
        self.top_k = top_k
        self.model_name = model_name
        self.model = None
        self.tokenizer = None
        self.device = "cuda" if torch and torch.cuda.is_available() else "cpu"
        
    def load_model(self):
        if self.model is not None:
            return
        if torch is None:
            print("[Cảnh báo] Chưa cài đặt transformers/torch.")
            return
        print(f"Đang nạp mô hình Reranker: {self.model_name} trên {self.device}...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, padding_side="left")
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                low_cpu_mem_usage=True
            ).eval().to(self.device)
            
            self.token_false_id = self.tokenizer.convert_tokens_to_ids("no")
            self.token_true_id = self.tokenizer.convert_tokens_to_ids("yes")
            self.max_length = 8192
            
            self.prefix = "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n"
            self.suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
            
            self.prefix_tokens = self.tokenizer.encode(self.prefix, add_special_tokens=False)
            self.suffix_tokens = self.tokenizer.encode(self.suffix, add_special_tokens=False)
            print("Đã nạp Cross-Encoder thành công!")
        except Exception as e:
            print(f"[Reranker] Lỗi nạp model: {str(e)}")
            self.model = None
            
    def _format_instruction(self, query: str, doc: str) -> str:
        instruction = "Given a web search query, retrieve relevant passages that answer the query"
        return f"<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"
        
    def _process_inputs(self, pairs: List[str]):
        # return_attention_mask=False để tránh lệch kích thước mask khi bọc token
        inputs = self.tokenizer(
            pairs,
            padding=False,
            truncation=True,
            return_attention_mask=False,
            max_length=self.max_length - len(self.prefix_tokens) - len(self.suffix_tokens)
        )
        for i, ele in enumerate(inputs["input_ids"]):
            inputs["input_ids"][i] = self.prefix_tokens + ele + self.suffix_tokens
        inputs = self.tokenizer.pad(inputs, padding=True, return_tensors="pt")
        for key in inputs:
            inputs[key] = inputs[key].to(self.device)
        return inputs
        
    def _compute_scores(self, inputs) -> List[float]:
        if torch is None or self.model is None:
            return [0.0] * len(inputs["input_ids"])
        with torch.no_grad():
            batch_scores = self.model(**inputs).logits[:, -1, :]
            true_vector = batch_scores[:, self.token_true_id]
            false_vector = batch_scores[:, self.token_false_id]
            batch_scores = torch.stack([false_vector, true_vector], dim=1)
            batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
            scores = batch_scores[:, 1].exp().tolist()
            return scores

    def rerank(self, query: str, documents: List[Document]) -> List[Document]:
        if not documents:
            return []
        self.load_model()
        if self.model is None:
            return documents[:self.top_k]
        try:
            pairs = [self._format_instruction(query, doc.page_content) for doc in documents]
            inputs = self._process_inputs(pairs)
            scores = self._compute_scores(inputs)
            doc_score_pairs = list(zip(documents, scores))
            doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
            return [doc for doc, score in doc_score_pairs[:self.top_k]]
        except Exception as e:
            print(f"[Reranker] Lỗi thực thi: {str(e)}")
            return documents[:self.top_k]

class MMRReranker:
    def __init__(self, embedding_model: Any, top_k: int = 3, lambda_mult: float = 0.5):
        self.embeddings = embedding_model
        self.top_k = top_k
        self.lambda_mult = lambda_mult
        
    def _calculate_cosine_similarity(self, v1, v2) -> float:
        norm1 = np.linalg.norm(v1)
        norm2 = np.linalg.norm(v2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return float(np.dot(v1, v2) / (norm1 * norm2))
        
    def rerank(self, query: str, documents: List[Document]) -> List[Document]:
        if not documents or len(documents) <= self.top_k:
            return documents
        try:
            query_embedding = np.array(self.embeddings.embed_query(query))
            doc_embeddings = [np.array(self.embeddings.embed_query(doc.page_content)) for doc in documents]
            
            selected_indices = []
            remaining_indices = list(range(len(documents)))
            
            similarities_to_query = [self._calculate_cosine_similarity(query_embedding, d_emb) for d_emb in doc_embeddings]
            best_idx = int(np.argmax(similarities_to_query))
            selected_indices.append(best_idx)
            remaining_indices.remove(best_idx)
            
            while len(selected_indices) < self.top_k and remaining_indices:
                best_mmr = -float("inf")
                best_candidate_idx = -1
                for candidate_idx in remaining_indices:
                    sim_to_query = similarities_to_query[candidate_idx]
                    max_sim_to_selected = max(
                        self._calculate_cosine_similarity(doc_embeddings[candidate_idx], doc_embeddings[selected_idx])
                        for selected_idx in selected_indices
                    )
                    mmr_score = self.lambda_mult * sim_to_query - (1 - self.lambda_mult) * max_sim_to_selected
                    if mmr_score > best_mmr:
                        best_mmr = mmr_score
                        best_candidate_idx = candidate_idx
                selected_indices.append(best_candidate_idx)
                remaining_indices.remove(best_candidate_idx)
            return [documents[idx] for idx in selected_indices]
        except Exception as e:
            print(f"[MMR] Lỗi: {str(e)}")
            return documents[:self.top_k]


### 🔗 Bước 8: Kết nối Advanced RAG Chain & Hậu xử lý kết quả
Đồng bộ các thành phần nạp dữ liệu, định tuyến, nén và sinh kết quả. Sử dụng `FocusedAnswerParser` biên dịch biểu mẫu trống.


In [ ]:
class FocusedAnswerParser:
    _BULLET_PATTERN = re.compile(r"^\s*[\u2022\-\*]\s*", re.MULTILINE)
    _NEWLINE_PATTERN = re.compile(r"\n+")

    @staticmethod
    def parse(text: str) -> str:
        if not text:
            return ""
        text = text.strip()
        if "[TRẢ LỜI]:" in text:
            text = text.split("[TRẢ LỜI]:")[-1].strip()
        elif "Trả lời:" in text:
            text = text.split("Trả lời:")[-1].strip()
            
        text = FocusedAnswerParser._BULLET_PATTERN.sub("", text)
        text = FocusedAnswerParser._NEWLINE_PATTERN.sub(" ", text).strip()
        return text

class RAGLogger:
    def __init__(self, log_dir: str = "./logs"):
        self.log_file = os.path.join(log_dir, "rag_traces.json")
        os.makedirs(log_dir, exist_ok=True)
        
    def log_trace(
        self, query: str, answer: str, contexts: List[str], 
        metadata_sources: List[Dict[str, Any]], execution_time: float, route: str, ground_truth: str = ""
    ):
        trace = {
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "question": query,
            "answer": answer,
            "contexts": contexts,
            "sources": metadata_sources,
            "execution_time_seconds": execution_time,
            "route": route,
            "ground_truth": ground_truth
        }
        traces = []
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, "r", encoding="utf-8") as f:
                    traces = json.load(f)
            except Exception:
                traces = []
        traces.append(trace)
        try:
            with open(self.log_file, "w", encoding="utf-8") as f:
                json.dump(traces, f, indent=4, ensure_ascii=False)
        except Exception as e:
            print(f"[Logger] Lỗi ghi log: {str(e)}")

    def get_all_traces(self) -> List[Dict[str, Any]]:
        if not os.path.exists(self.log_file):
            return []
        try:
            with open(self.log_file, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return []

class AdvancedRAGChain:
    def __init__(self, llm: BaseChatModel, embeddings: Any, vector_manager: Any, log_dir: str = "./logs"):
        self.llm = llm
        self.embeddings = embeddings
        self.vector_manager = vector_manager
        
        self.router = QueryRouter(llm)
        self.hyde_transformer = HyDEQueryTransformer(llm)
        self.decomp_transformer = DecompositionQueryTransformer(llm)
        
        self.cross_encoder_reranker = CrossEncoderReranker(top_k=3)
        self.mmr_reranker = MMRReranker(embedding_model=embeddings, top_k=3)
        self.web_search = WebSearchRetriever(max_results=5)
        self.logger = RAGLogger(log_dir)
        
        # Cache BM25 index để tránh rebuild tốn kém trên mỗi request
        self._bm25_retriever_cache: Optional[BM25Retriever] = None
        self._bm25_cache_doc_count: int = 0
        
        self.prompt_template = """Bạn là trợ lý AI chuyên nghiệp phân tích tài liệu tiếng Việt.
Hãy trả lời câu hỏi dựa trên các tài liệu được cung cấp dưới đây.

[TÀI LIỆU]:
{context}

[CÂU HỎI]:
{question}

[YÊU CẦU]:
- Trả lời trung thực, ngắn gọn và trực tiếp vào câu hỏi trong khoảng 3 đến 5 câu chi tiết.
- Không lặp lại câu hỏi, không chào hỏi dông dài, không tự tạo thông tin ngoài tài liệu được cung cấp.
- Nếu tài liệu không chứa câu trả lời, hãy nói rõ: "Không có thông tin trong tài liệu".

[TRẢ LỜI]:"""
        self.prompt = PromptTemplate.from_template(self.prompt_template)

    def _format_docs(self, docs: List[Document]) -> str:
        formatted = []
        seen = set()
        for doc in docs:
            content = doc.page_content.strip()
            if content and len(content) > 40 and content not in seen:
                formatted.append(content)
                seen.add(content)
        return "\n\n".join(formatted)

    def run(self, query: str, config: Dict[str, Any]) -> Dict[str, Any]:
        start_time = time.time()
        execution_logs = {}
        
        search_mode = config.get("search_mode", "hybrid")
        fusion_mode = config.get("fusion_mode", "rrf")
        query_transform = config.get("query_transform", "none")
        reranker_type = config.get("reranker_type", "none")
        use_router = config.get("use_router", True)
        force_internet = config.get("force_internet", False)
        selected_tags = config.get("doc_tags", [])
        
        # Bước 1: Routing
        route = "local"
        if use_router and not force_internet:
            route = self.router.route_query(query)
            execution_logs["route_decision"] = route
        elif force_internet:
            route = "web"
            execution_logs["route_decision"] = "web (forced)"
            
        # Bước 2: Retrieval
        retrieved_documents = []
        vector_retriever = self.vector_manager.get_retriever(k=10, doc_tags=selected_tags)
        
        bm25_retriever = None
        try:
            db_data = self.vector_manager.db.get()
            raw_docs = db_data["documents"]
            metadatas = db_data["metadatas"]
            current_doc_count = len(raw_docs)
            
            if self._bm25_retriever_cache is None or self._bm25_cache_doc_count != current_doc_count:
                all_docs_for_bm25 = [
                    Document(page_content=text, metadata=meta)
                    for text, meta in zip(raw_docs, metadatas)
                    if text.strip()
                ]
                if all_docs_for_bm25:
                    self._bm25_retriever_cache = BM25Retriever.from_documents(all_docs_for_bm25, k=10)
                    self._bm25_cache_doc_count = current_doc_count
            bm25_retriever = self._bm25_retriever_cache
        except Exception as e:
            print(f"[BM25] Lỗi khởi tạo: {str(e)}")
            bm25_retriever = None

        if route == "web":
            retrieved_documents = self.web_search.invoke(query)
            execution_logs["retrieval_source"] = "DuckDuckGo Web Search"
        else:
            search_queries = [query]
            if query_transform == "hyde":
                hyde_doc = self.hyde_transformer.generate_hypothetical_doc(query)
                search_queries = [hyde_doc]
                execution_logs["hyde_document"] = hyde_doc
            elif query_transform == "decomposition":
                sub_queries = self.decomp_transformer.decompose(query)
                search_queries = sub_queries + [query]
                execution_logs["decomposed_queries"] = sub_queries

            local_docs = []
            seen_content = set()
            for q in search_queries:
                q_docs = []
                if search_mode == "dense":
                    q_docs = vector_retriever.invoke(q)
                elif search_mode == "sparse" and bm25_retriever:
                    q_docs = bm25_retriever.invoke(q)
                elif search_mode == "hybrid" and bm25_retriever:
                    hybrid_engine = HybridRetriever(
                        bm25_retriever=bm25_retriever,
                        vector_retriever=vector_retriever,
                        top_k=10,
                        fusion_mode=fusion_mode
                    )
                    q_docs = hybrid_engine.invoke(q)
                else:
                    q_docs = vector_retriever.invoke(q)
                    
                for doc in q_docs:
                    if doc.page_content not in seen_content:
                        local_docs.append(doc)
                        seen_content.add(doc.page_content)
            retrieved_documents = local_docs
            execution_logs["retrieval_source"] = "Local Knowledge Base"
            
            if route == "hybrid" or (len(retrieved_documents) == 0 and use_router):
                web_docs = self.web_search.invoke(query)
                for doc in web_docs:
                    if doc.page_content not in seen_content:
                        retrieved_documents.append(doc)
                        seen_content.add(doc.page_content)
                execution_logs["retrieval_source"] += " + Web Search"

        # Bước 3: Context Compression / Reranking
        reranked_documents = retrieved_documents
        if reranker_type == "cross_encoder":
            reranked_documents = self.cross_encoder_reranker.rerank(query, retrieved_documents)
        elif reranker_type == "mmr":
            reranked_documents = self.mmr_reranker.rerank(query, retrieved_documents)
        else:
            reranked_documents = retrieved_documents[:3]
            
        execution_logs["raw_retrieved_count"] = len(retrieved_documents)
        execution_logs["reranked_count"] = len(reranked_documents)

        # Bước 4: Generation
        context_str = self._format_docs(reranked_documents)
        if not context_str.strip():
            context_str = "Không có tài liệu phù hợp được tìm thấy."
            
        formatted_prompt = self.prompt.format(context=context_str, question=query)
        try:
            response = self.llm.invoke(formatted_prompt)
            raw_answer = response.content.strip()
        except Exception as e:
            raw_answer = f"[Lỗi hệ thống sinh câu trả lời]: {str(e)}"
            
        final_answer = FocusedAnswerParser.parse(raw_answer)
        
        citations = []
        for doc in reranked_documents:
            meta = doc.metadata
            citations.append({
                "source": meta.get("source_name", "Tài liệu không rõ nguồn"),
                "tag": meta.get("doc_tag", "Chưa phân loại"),
                "page": meta.get("page", 1),
                "snippet": doc.page_content[:150] + "..."
            })
            
        elapsed_time = time.time() - start_time
        
        # Log trace
        self.logger.log_trace(
            query=query, answer=final_answer, contexts=[doc.page_content for doc in reranked_documents],
            metadata_sources=citations, execution_time=elapsed_time, route=route
        )
        return {
            "query": query, "answer": final_answer, "citations": citations,
            "execution_time_seconds": elapsed_time, "logs": execution_logs, "context_used": context_str
        }


### 🎯 Bước 9: Triển khai EmbeddingFinetuner với Đánh giá đầu ra
Tính toán chỉ số **Hit Rate@5** và **MRR@5** trước và sau khi Fine-tune ( SentenceTransformer + MultipleNegativesRankingLoss ).


In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

class EmbeddingFinetuner:
    def __init__(self, llm: BaseChatModel, base_model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
        self.llm = llm
        self.base_model_name = base_model_name
        self.output_dir = "./models/fine_tuned_embeddings"
        
        self.question_generator_prompt = """Bạn là một chuyên gia khảo thí. Hãy đọc đoạn văn dưới đây và viết đúng 1 câu hỏi duy nhất mà đoạn văn này có thể trả lời trực tiếp một cách đầy đủ.
Yêu cầu câu hỏi tự nhiên bằng tiếng Việt, đi thẳng vào ý chính, không chứa thông tin thừa.
Không viết thêm giải thích nào khác.

Đoạn văn:
"{context}"

Câu hỏi tiếng Việt:"""

    def generate_synthetic_dataset(self, chunks: List[Document], max_samples: int = 50) -> List[Tuple[str, str]]:
        print(f"Tạo synthetic dataset từ {min(len(chunks), max_samples)} chunks...")
        training_pairs = []
        sampled_chunks = chunks[:max_samples]
        
        for idx, chunk in enumerate(tqdm(sampled_chunks, desc="Generating questions")):
            context = chunk.page_content
            prompt = self.question_generator_prompt.format(context=context[:800])
            try:
                response = self.llm.invoke(prompt)
                question = response.content.strip().replace('"', '').replace("'", "").strip()
                if len(question) > 10:
                    training_pairs.append((question, context))
                time.sleep(0.5)
            except Exception as e:
                print(f"[Lỗi] Chunk {idx}: {str(e)}")
        return training_pairs

    def run_finetuning(self, training_pairs: List[Tuple[str, str]], epochs: int = 1, batch_size: int = 8) -> str:
        if not training_pairs:
            raise ValueError("Training dataset trống.")
        print(f"Khởi tạo mô hình nền: {self.base_model_name}...")
        model = SentenceTransformer(self.base_model_name)
        
        train_examples = [InputExample(texts=[q, c]) for q, c in training_pairs]
        train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
        train_loss = losses.MultipleNegativesRankingLoss(model)
        
        print(f"Huấn luyện tinh chỉnh (Epochs={epochs}, Batch={batch_size})...")
        os.makedirs(self.output_dir, exist_ok=True)
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=epochs,
            show_progress_bar=True
        )
        model.save(self.output_dir)
        return self.output_dir

    def evaluate_embeddings(self, model_or_path: Any, eval_pairs: List[Tuple[str, str]], k: int = 5) -> Dict[str, float]:
        if not eval_pairs:
            return {"hit_rate": 0.0, "mrr": 0.0}
        if isinstance(model_or_path, str):
            model = SentenceTransformer(model_or_path)
        else:
            model = model_or_path
            
        try:
            questions = [pair[0] for pair in eval_pairs]
            corpus = list(set([pair[1] for pair in eval_pairs]))
            
            q_embs = model.encode(questions, show_progress_bar=False)
            c_embs = model.encode(corpus, show_progress_bar=False)
            
            q_norms = np.linalg.norm(q_embs, axis=1, keepdims=True)
            c_norms = np.linalg.norm(c_embs, axis=1, keepdims=True)
            q_norms[q_norms == 0] = 1e-10
            c_norms[c_norms == 0] = 1e-10
            
            q_embs = q_embs / q_norms
            c_embs = c_embs / c_norms
            
            similarity_matrix = np.dot(q_embs, c_embs.T)
            hits = 0
            mrr_sum = 0.0
            
            for i, (q, correct_ctx) in enumerate(eval_pairs):
                correct_idx = corpus.index(correct_ctx)
                scores = similarity_matrix[i]
                sorted_indices = np.argsort(scores)[::-1]
                rank = np.where(sorted_indices == correct_idx)[0][0] + 1
                if rank <= k:
                    hits += 1
                    mrr_sum += 1.0 / rank
            return {
                "hit_rate": round(hits / len(eval_pairs), 4),
                "mrr": round(mrr_sum / len(eval_pairs), 4)
            }
        except Exception as e:
            print(f"[Đánh giá] Lỗi: {str(e)}")
            return {"hit_rate": 0.0, "mrr": 0.0}


### 📊 Bước 10: Triển khai RAGEvaluator tự động phân loại metric
Tránh lỗi khoa học "tự đá bóng tự thổi còi". Tự động định tuyến sang bộ metric **Reference-Free** (Faithfulness, Answer Relevancy) khi không có đáp án chuẩn con người.


In [ ]:
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    _RAGAS_AVAILABLE = True
except ImportError:
    _RAGAS_AVAILABLE = False

class RAGEvaluator:
    def __init__(self, output_dir: str = "./logs"):
        self.report_file = os.path.join(output_dir, "eval_report.json")
        os.makedirs(output_dir, exist_ok=True)
        
    def _has_ground_truth(self, test_dataset: List[Dict[str, Any]]) -> bool:
        gt_count = sum(1 for item in test_dataset if item.get("ground_truth", "").strip())
        return gt_count >= len(test_dataset) * 0.5
        
    def run_evaluation(
        self, evaluator_llm: Any, evaluator_embeddings: Any, test_dataset: List[Dict[str, Any]] = None
    ) -> Dict[str, Any]:
        if not test_dataset:
            raise ValueError("Test dataset trống.")
        if not _RAGAS_AVAILABLE:
            raise ImportError("Vui lòng cài đặt ragas: pip install ragas")
            
        use_reference_metrics = self._has_ground_truth(test_dataset)
        if use_reference_metrics:
            metrics_to_run = [faithfulness, answer_relevancy, context_precision, context_recall]
            eval_mode = "Full (Reference-Based): Faithfulness + Answer Relevancy + Context Precision + Context Recall"
        else:
            metrics_to_run = [faithfulness, answer_relevancy]
            eval_mode = "Reference-Free: Faithfulness + Answer Relevancy (Không có ground_truth)"
            
        print(f"Chạy Ragas Evaluation: {eval_mode}...")
        
        data_dict = {
            "question": [item["question"] for item in test_dataset],
            "answer": [item["answer"] for item in test_dataset],
            "contexts": [item["contexts"] for item in test_dataset],
            "ground_truth": [item.get("ground_truth", "") for item in test_dataset]
        }
        dataset = Dataset.from_dict(data_dict)
        wrapped_llm = LangchainLLMWrapper(evaluator_llm)
        wrapped_embeddings = LangchainEmbeddingsWrapper(evaluator_embeddings)
        
        try:
            results = evaluate(
                dataset=dataset, metrics=metrics_to_run, llm=wrapped_llm,
                embeddings=wrapped_embeddings, raise_exceptions=False
            )
            scores = dict(results)
            report = {
                "eval_timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
                "samples_evaluated": len(test_dataset),
                "eval_mode": eval_mode,
                "has_ground_truth": use_reference_metrics,
                "scores": scores,
                "status": "success"
            }
            with open(self.report_file, "w", encoding="utf-8") as f:
                json.dump(report, f, indent=4, ensure_ascii=False)
            return report
        except Exception as e:
            print(f"[Ragas] Lỗi: {str(e)}")
            return {"status": "failed", "error": str(e)}


### 🚀 Bước 11: Khởi chạy Giao diện Gradio tương tác trực tiếp
Thiết lập Gradio Block hỗ trợ Dark Mode và sinh liên kết share link dùng trên Colab.


In [ ]:
# File app.py được tích hợp trực tiếp để khởi chạy
import gradio as gr
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

def get_active_embeddings(use_finetuned: bool = True):
    finetuned_path = os.path.join(MODELS_DIR, "fine_tuned_embeddings")
    if use_finetuned and os.path.exists(os.path.join(finetuned_path, "model.safetensors")) or os.path.exists(os.path.join(finetuned_path, "pytorch_model.bin")):
        print(f"[Active Embedding] Tải mô hình Fine-tuned: {finetuned_path}")
        return HuggingFaceEmbeddings(model_name=finetuned_path)
    print("[Active Embedding] Tải mô hình mặc định: paraphrase-multilingual-MiniLM-L12-v2")
    return HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

embeddings = get_active_embeddings()
vector_manager = VectorStoreManager(persist_directory=CHROMA_DIR, embedding_model=embeddings)
try:
    vector_manager.init_db()
except Exception:
    pass

# Helper functions for UI
def list_indexed_documents():
    try:
        data = vector_manager.db.get()
        metadatas = data["metadatas"]
        if not metadatas:
            return pd.DataFrame(columns=["Tên File", "Nhãn Tag", "Số Chunks"])
        file_counts = {}
        file_tags = {}
        for m in metadatas:
            file_name = m.get("source_name", "Không rõ")
            tag = m.get("doc_tag", "Chưa phân loại")
            file_counts[file_name] = file_counts.get(file_name, 0) + 1
            file_tags[file_name] = tag
        df_list = [{"Tên File": name, "Nhãn Tag": file_tags[name], "Số Chunks": count} for name, count in file_counts.items()]
        return pd.DataFrame(df_list)
    except Exception:
        return pd.DataFrame(columns=["Tên File", "Nhãn Tag", "Số Chunks"])

def upload_file_interface(files, custom_tag):
    if not files: return "Vui lòng chọn file.", list_indexed_documents()
    tag = custom_tag.strip() or "Upload"
    loader = SimpleLoader()
    chunker = SemanticChunker(embedding_model=embeddings, breakpoint_threshold=0.5, max_chunk_tokens=350)
    total_chunks = 0
    try:
        for file in files:
            raw_docs = loader.load_file(file.name, tag=tag)
            chunks = chunker.split(raw_docs)
            if chunks:
                vector_manager.db.add_documents(chunks)
                total_chunks += len(chunks)
        return f"Đã nạp thành công {len(files)} files với {total_chunks} chunks!", list_indexed_documents()
    except Exception as e:
        return f"Lỗi nạp file: {str(e)}", list_indexed_documents()

def rag_qa_interface(
    query, llm_provider, api_key, chunker_type, search_mode, fusion_mode, query_transform, reranker_type, use_router, force_internet, selected_tags
):
    if not query.strip(): return "Nhập câu hỏi.", "", "Không trích dẫn."
    try:
        if llm_provider == "Gemini API":
            os.environ["GEMINI_API_KEY"] = api_key or os.environ.get("GEMINI_API_KEY", "")
            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", api_version="v1", temperature=0.2)
        else:
            os.environ["OPENAI_API_KEY"] = api_key or os.environ.get("OPENAI_API_KEY", "")
            llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
        
        config = {
            "search_mode": search_mode, "fusion_mode": fusion_mode, "query_transform": query_transform,
            "reranker_type": reranker_type, "use_router": use_router, "force_internet": force_internet, "doc_tags": selected_tags
        }
        chain = AdvancedRAGChain(llm=llm, embeddings=embeddings, vector_manager=vector_manager, log_dir=LOG_DIR)
        result = chain.run(query, config)
        
        answer = result["answer"]
        logs = result["logs"]
        citations = result["citations"]
        
        log_html = f"<b>Route:</b> {logs.get('route_decision', 'N/A')}<br><b>Source:</b> {logs.get('retrieval_source', 'N/A')}<br>"
        log_html += f"<b>Raw Chunks:</b> {logs.get('raw_retrieved_count', 0)} | <b>Reranked Chunks:</b> {logs.get('reranked_count', 0)}<br>"
        
        citations_html = ""
        for idx, c in enumerate(citations):
            citations_html += f"""
            <div style='border: 1px solid #444; padding: 8px; margin-bottom: 6px; background-color: #1e1e1e; border-radius: 4px;'>
                <span style='color: #4CAF50;'><b>[Nguồn {idx+1}] {c['source']}</b></span> (Trang {c['page']})<br>
                <span style='color: #ccc; font-style: italic;'>"{c['snippet']}"</span>
            </div>"""
        return answer, log_html, citations_html
    except Exception as e:
        return f"Lỗi: {str(e)}", "", ""

def run_finetuning_ui(llm_provider, api_key, sample_size, epochs, batch_size):
    global embeddings, vector_manager
    try:
        if llm_provider == "Gemini API":
            os.environ["GEMINI_API_KEY"] = api_key or os.environ.get("GEMINI_API_KEY", "")
            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", api_version="v1")
        else:
            os.environ["OPENAI_API_KEY"] = api_key or os.environ.get("OPENAI_API_KEY", "")
            llm = ChatOpenAI(model="gpt-4o-mini")
            
        data = vector_manager.db.get()
        raw_texts, metadatas = data["documents"], data["metadatas"]
        if not raw_texts: return "Cơ sở dữ liệu trống, vui lòng tải tài liệu lên trước."
        chunks = [Document(page_content=t, metadata=m) for t, m in zip(raw_texts, metadatas)]
        
        finetuner = EmbeddingFinetuner(llm=llm)
        training_pairs = finetuner.generate_synthetic_dataset(chunks, max_samples=int(sample_size))
        
        random.seed(SEED)
        random.shuffle(training_pairs)
        split_idx = int(len(training_pairs) * 0.8)
        train_pairs = training_pairs[:split_idx]
        test_pairs = training_pairs[split_idx:] if split_idx < len(training_pairs) else training_pairs
        
        # Eval Base Model
        base_model = SentenceTransformer(finetuner.base_model_name)
        pre_eval = finetuner.evaluate_embeddings(base_model, test_pairs, k=5)
        
        # Run Training
        model_path = finetuner.run_finetuning(train_pairs, epochs=int(epochs), batch_size=int(batch_size))
        
        # Eval Fine-tuned Model
        post_eval = finetuner.evaluate_embeddings(model_path, test_pairs, k=5)
        
        # Reload
        embeddings = HuggingFaceEmbeddings(model_name=model_path)
        vector_manager = VectorStoreManager(persist_directory=CHROMA_DIR, embedding_model=embeddings)
        vector_manager.init_db()
        
        report = f"🎉 Huấn luyện thành công! Mô hình lưu tại {model_path}\n\n"
        report += f"📊 ĐÁNH GIÁ ĐẦU RA EMBEDDING (Trên {len(test_pairs)} mẫu test độc lập):\n"
        report += f"- TRƯỚC KHI TINH CHỈNH:\n  • Hit Rate@5: {pre_eval['hit_rate']*100:.2f}% | MRR@5: {pre_eval['mrr']*100:.2f}%\n"
        report += f"- SAU KHI TINH CHỈNH:\n  • Hit Rate@5: {post_eval['hit_rate']*100:.2f}% | MRR@5: {post_eval['mrr']*100:.2f}%\n\n"
        report += f"📈 Hiệu năng thay đổi: Hit Rate: {(post_eval['hit_rate']-pre_eval['hit_rate'])*100:+.2f}% | MRR: {(post_eval['mrr']-pre_eval['mrr'])*100:+.2f}%"
        return report
    except Exception as e:
        return f"Lỗi: {str(e)}"

# Setup Gradio Interface
custom_css = """
body { background-color: #121212; color: #e0e0e0; font-family: 'Inter', sans-serif; }
.gradio-container { max-width: 1200px !important; margin: 0 auto !important; }
.btn-primary { background-color: #4CAF50 !important; color: white !important; }
"""
with gr.Blocks(title="Advanced Vietnamese RAG System") as demo:
    gr.Markdown("# 🧠 Hệ thống Hỏi Đáp Tài Liệu Tiếng Việt Nâng Cao (Advanced RAG)")
    
    with gr.Row():
        llm_provider = gr.Dropdown(choices=["Gemini API", "OpenAI API"], value="Gemini API", label="LLM Provider")
        api_key = gr.Textbox(type="password", placeholder="Nhập API Key ở đây (hoặc cấu hình ENV)", label="API Key")
        
    with gr.Tabs():
        with gr.Tab("💬 Hỏi Đáp RAG"):
            with gr.Row():
                with gr.Column(scale=1):
                    use_router = gr.Checkbox(value=True, label="Kích hoạt Bộ định tuyến (Query Router)")
                    force_internet = gr.Checkbox(value=False, label="Ép buộc Tra cứu Internet (Web Search)")
                    chunker_type = gr.Radio(choices=["Fixed Size Chunker", "Semantic Chunker"], value="Semantic Chunker", label="1. Chunker")
                    search_mode = gr.Radio(choices=["dense", "sparse", "hybrid"], value="hybrid", label="2. Retrieval Mode")
                    fusion_mode = gr.Radio(choices=["rrf", "interleave"], value="rrf", label="3. Fusion Mode")
                    query_transform = gr.Radio(choices=["none", "hyde", "decomposition"], value="none", label="4. Query Transform")
                    reranker_type = gr.Radio(choices=["none", "cross_encoder", "mmr"], value="none", label="5. Reranker")
                    selected_tags = gr.CheckboxGroup(choices=["AI", "LuatDatDai", "LuatLaoDong", "Upload"], value=[], label="🏷️ Lọc theo nhãn (Pre-filtering)")
                with gr.Column(scale=2):
                    query_input = gr.Textbox(lines=3, placeholder="Nhập câu hỏi...", label="Câu hỏi:")
                    submit_btn = gr.Button("Gửi câu hỏi", variant="primary")
                    answer_output = gr.Markdown(value="Câu trả lời hiển thị ở đây...")
                    with gr.Accordion("🔍 Nhật ký xử lý & Ý định RAG", open=False):
                        log_output = gr.HTML(value="Chi tiết...")
                    citations_output = gr.HTML(value="Nguồn trích dẫn...")
            submit_btn.click(
                fn=rag_qa_interface,
                inputs=[query_input, llm_provider, api_key, chunker_type, search_mode, fusion_mode, query_transform, reranker_type, use_router, force_internet, selected_tags],
                outputs=[answer_output, log_output, citations_output]
            )
            
        with gr.Tab("📁 Quản Lý Tài Liệu"):
            with gr.Row():
                with gr.Column():
                    uploaded_files = gr.File(file_count="multiple", file_types=[".pdf", ".docx"], label="Tải file PDF/DOCX")
                    custom_tag = gr.Textbox(value="Upload", label="Nhãn Tag")
                    upload_btn = gr.Button("Thêm tài liệu", variant="primary")
                with gr.Column():
                    doc_table = gr.Dataframe(value=list_indexed_documents(), interactive=False)
            upload_btn.click(fn=upload_file_interface, inputs=[uploaded_files, custom_tag], outputs=[gr.Textbox(label="Thông báo"), doc_table])
            
        with gr.Tab("📊 Huấn luyện"):
            with gr.Row():
                with gr.Column():
                    sample_size = gr.Slider(minimum=10, maximum=150, value=30, step=5, label="Sample Size")
                    epochs = gr.Slider(minimum=1, maximum=5, value=1, step=1, label="Epochs")
                    batch_size = gr.Slider(minimum=4, maximum=32, value=8, step=4, label="Batch Size")
                    run_ft_btn = gr.Button("Huấn luyện & Đánh giá", variant="primary")
                with gr.Column():
                    ft_status = gr.Textbox(value="Chưa khởi chạy...", label="Kết quả")
            run_ft_btn.click(fn=run_finetuning_ui, inputs=[llm_provider, api_key, sample_size, epochs, batch_size], outputs=[ft_status])

demo.launch(share=True, css=custom_css, prevent_thread_lock=True)
